# Feature Engineering in Python (Beginner Guide)
Feature engineering = making your data better for analysis and machine learning.
Run each cell one by one. Read the comments (lines starting with #).

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   OneHotEncoder, OrdinalEncoder, PolynomialFeatures)
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

## 2. Create sample data
We make fake customer data so you can practice.

In [ ]:
np.random.seed(42)
n = 200
df = pd.DataFrame({
    "customer_id": range(1, n + 1),
    "age": np.random.randint(18, 70, n).astype(float),
    "income": np.random.normal(50000, 15000, n).round(0),
    "city": np.random.choice(["Delhi", "Mumbai", "Chennai", "Pune"], n),
    "education": np.random.choice(["School", "Bachelor", "Master", "PhD"], n),
    "signup_date": pd.to_datetime("2023-01-01") + pd.to_timedelta(np.random.randint(0, 700, n), unit="D"),
    "total_orders": np.random.randint(1, 50, n),
    "total_spent": np.random.exponential(2000, n).round(2),
    "review": np.random.choice(["good product", "bad service", "very good and fast", "not happy", "ok"], n),
    "churn": np.random.randint(0, 2, n),
})
# Add problems on purpose: missing values, outlier, duplicate row
df.loc[[3, 10, 25], "age"] = np.nan
df.loc[[5, 40], "income"] = np.nan
df.loc[[7, 50], "city"] = np.nan
df.loc[15, "income"] = 900000          # outlier
df = pd.concat([df, df.iloc[[0]]], ignore_index=True)  # duplicate row
df.head()

## 3. Look at the data first

In [ ]:
print(df.shape)          # rows, columns
print(df.dtypes)         # data types
print(df.isnull().sum()) # missing values per column
df.describe()            # summary of numbers

## 4. Handle missing values
Numbers -> fill with median. Text -> fill with most common value (mode).

In [ ]:
df["age"] = df["age"].fillna(df["age"].median())
df["income"] = df["income"].fillna(df["income"].median())
df["city"] = df["city"].fillna(df["city"].mode()[0])

# Extra idea: a flag column that says "this value was missing"
# df["age_missing"] = df["age"].isnull().astype(int)   # (run BEFORE filling)
print(df.isnull().sum())

## 5. Remove duplicates

In [ ]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("New shape:", df.shape)

## 6. Handle outliers (IQR method)
Values too far from the middle are outliers. We cap them (clip).

In [ ]:
Q1 = df["income"].quantile(0.25)
Q3 = df["income"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
print("Outliers found:", ((df["income"] < lower) | (df["income"] > upper)).sum())
df["income"] = df["income"].clip(lower, upper)

## 7. Date features
One date column can give many useful columns.

In [ ]:
df["signup_year"] = df["signup_date"].dt.year
df["signup_month"] = df["signup_date"].dt.month
df["signup_day"] = df["signup_date"].dt.day
df["signup_weekday"] = df["signup_date"].dt.dayofweek      # 0 = Monday
df["is_weekend"] = (df["signup_weekday"] >= 5).astype(int)
df["signup_quarter"] = df["signup_date"].dt.quarter
# How many days since signup (customer age in days)
today = pd.Timestamp("2025-01-01")
df["days_since_signup"] = (today - df["signup_date"]).dt.days
df[["signup_date", "signup_month", "is_weekend", "days_since_signup"]].head()

## 8. Create new numeric features
Mix columns to make new meaning.

In [ ]:
# Ratio: average money per order
df["avg_order_value"] = df["total_spent"] / df["total_orders"]
# Income per age
df["income_per_age"] = df["income"] / df["age"]
# Orders per day since signup
df["orders_per_day"] = df["total_orders"] / df["days_since_signup"]
df[["avg_order_value", "income_per_age", "orders_per_day"]].head()

## 9. Transformations (fix skewed data)
total_spent is skewed (many small, few big). Log makes it more balanced.

In [ ]:
df["total_spent_log"] = np.log1p(df["total_spent"])     # log(1 + x)
df["total_spent_sqrt"] = np.sqrt(df["total_spent"])
print("Skew before:", round(df["total_spent"].skew(), 2))
print("Skew after log:", round(df["total_spent_log"].skew(), 2))

## 10. Binning (make groups from numbers)

In [ ]:
# Fixed groups
df["age_group"] = pd.cut(df["age"], bins=[0, 25, 35, 50, 100],
                         labels=["Young", "Adult", "Middle", "Senior"])
# Equal-size groups (quantiles)
df["income_level"] = pd.qcut(df["income"], q=4, labels=["Low", "Mid", "High", "Very High"])
df[["age", "age_group", "income", "income_level"]].head()

## 11. Scaling (put numbers on the same scale)
Important for distance-based models (KNN, SVM, PCA, linear models).

In [ ]:
cols = ["age", "income", "total_spent"]
df["age_std"] = StandardScaler().fit_transform(df[["age"]])         # mean 0, std 1
df["income_minmax"] = MinMaxScaler().fit_transform(df[["income"]])  # 0 to 1
df["spent_robust"] = RobustScaler().fit_transform(df[["total_spent"]])  # good with outliers
df[["age_std", "income_minmax", "spent_robust"]].describe().round(2)

## 12. Polynomial and interaction features

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_arr = poly.fit_transform(df[["age", "total_orders"]])
poly_df = pd.DataFrame(poly_arr, columns=poly.get_feature_names_out(["age", "total_orders"]))
poly_df.head()
# Columns: age, total_orders, age^2, age*total_orders, total_orders^2

## 13. Encode categorical data
Models need numbers, not text.

### 13a. Ordinal encoding (when order matters)

In [ ]:
order = [["School", "Bachelor", "Master", "PhD"]]
df["education_ord"] = OrdinalEncoder(categories=order).fit_transform(df[["education"]])
df[["education", "education_ord"]].drop_duplicates().sort_values("education_ord")

### 13b. One-hot encoding (when no order)

In [ ]:
df = pd.get_dummies(df, columns=["city"], prefix="city", drop_first=False, dtype=int)
df.filter(like="city_").head()

### 13c. Frequency encoding
Replace category with how many times it appears.

In [ ]:
freq = df["education"].value_counts(normalize=True)
df["education_freq"] = df["education"].map(freq)
df[["education", "education_freq"]].head()

### 13d. Target (mean) encoding
Replace category with average of the target. WARNING: can leak information.
Always calculate it on TRAIN data only in real projects.

In [ ]:
target_mean = df.groupby("education")["churn"].mean()
df["education_target"] = df["education"].map(target_mean)
df[["education", "education_target"]].drop_duplicates()

## 14. Simple text features

In [ ]:
df["review_length"] = df["review"].str.len()
df["review_words"] = df["review"].str.split().str.len()
df["has_good"] = df["review"].str.contains("good").astype(int)
df["has_not"] = df["review"].str.contains("not|bad").astype(int)
df[["review", "review_length", "review_words", "has_good", "has_not"]].head()

## 15. Group (aggregation) features
Add information about the group each row belongs to.

In [ ]:
df["edu_avg_income"] = df.groupby("education")["income"].transform("mean")
df["edu_max_spent"] = df.groupby("education")["total_spent"].transform("max")
df["income_vs_edu_avg"] = df["income"] - df["edu_avg_income"]
df[["education", "income", "edu_avg_income", "income_vs_edu_avg"]].head()

## 16. Feature selection (keep the useful features)

In [ ]:
num_df = df.select_dtypes(include="number").drop(columns=["customer_id", "churn"])
y = df["churn"]

# 16a. Remove features with almost no variation
vt = VarianceThreshold(threshold=0.01)
vt.fit(num_df)
print("Low variance removed:", list(num_df.columns[~vt.get_support()]))

# 16b. Remove highly correlated features (duplicates of each other)
corr = num_df.corr().abs()
upper_tri = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_cols = [c for c in upper_tri.columns if any(upper_tri[c] > 0.9)]
print("Highly correlated (drop):", drop_cols)

# 16c. Pick best K features using a statistical test
selector = SelectKBest(score_func=f_classif, k=5)
selector.fit(num_df, y)
print("Top 5 features:", list(num_df.columns[selector.get_support()]))

## 17. PCA (reduce many columns to few)

In [ ]:
scaled = StandardScaler().fit_transform(num_df.drop(columns=drop_cols))
pca = PCA(n_components=3)
pcs = pca.fit_transform(scaled)
print("Variance explained:", pca.explained_variance_ratio_.round(3))
pd.DataFrame(pcs, columns=["PC1", "PC2", "PC3"]).head()

## 18. Full Pipeline (the professional way)
Do all steps in one object. This avoids data leakage and is easy to reuse.

In [ ]:
raw = pd.DataFrame({
    "age": np.random.randint(18, 70, n).astype(float),
    "income": np.random.normal(50000, 15000, n),
    "total_orders": np.random.randint(1, 50, n),
    "city": np.random.choice(["Delhi", "Mumbai", "Chennai", "Pune"], n),
    "education": np.random.choice(["School", "Bachelor", "Master", "PhD"], n),
    "churn": np.random.randint(0, 2, n),
})
raw.loc[[1, 2, 3], "age"] = np.nan
raw.loc[[4, 5], "city"] = np.nan

X = raw.drop(columns="churn")
y = raw["churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = ["age", "income", "total_orders"]
cat_cols = ["city"]
ord_cols = ["education"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
    ("ord", OrdinalEncoder(categories=[["School", "Bachelor", "Master", "PhD"]]), ord_cols),
])

model = Pipeline([("prep", preprocess),
                  ("clf", LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
print("Test accuracy:", round(model.score(X_test, y_test), 3))

## 19. Summary cheat sheet
| Problem | Technique |
|---|---|
| Missing values | fillna, SimpleImputer |
| Outliers | IQR clip, RobustScaler |
| Dates | year, month, weekday, days since |
| Skewed numbers | log1p, sqrt |
| Numbers to groups | pd.cut, pd.qcut |
| Different scales | StandardScaler, MinMaxScaler |
| Text categories | Ordinal, One-hot, Frequency, Target |
| New ideas | ratios, interactions, group stats |
| Too many features | variance, correlation, SelectKBest, PCA |
| Clean workflow | ColumnTransformer + Pipeline |